In [13]:
import os
import torch
import torch.nn as nn
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim

cwd = Path(os.getcwd())
images_lib = cwd / "data" / "images" / "Images"
annotations_lib = cwd / "data" / "Annotation"

dtype = torch.float
device = torch.device("cuda:0")

In [14]:
# get all the labels

classes = []
for directory in os.listdir(annotations_lib):
    classes += [directory[10:]]

In [32]:
# to do: transform label into an index label (0-119)
# labels input needs to be a tensor of (N, C) where C is # of labels

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=20, kernel_size=3, stride=1, padding=1) 
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.conv2 = nn.Conv2d(in_channels=20, out_channels=16, kernel_size=3, stride=1, padding=1) 
        self.fc1 = nn.Linear(in_features=25*25*16*20, out_features=1000)
        self.fc2 = nn.Linear(in_features=1000, out_features=300)
        self.fc3 = nn.Linear(in_features=300, out_features=120*20)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) #in: 3x100x100, out: 20x100x100, pool: 20x50x50
        x = self.pool(F.relu(self.conv2(x))) #in: 20x50x50, out: 16x50x50, pool: 16x25x25
        x = x.view(1, 25*25*16*20) # in: 16x25x25, out: 1x10000
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        x = x.view(20, 120)
        m = nn.LogSoftmax()
        return m(x)

In [33]:
class labeled_image:
    def __init__(self, image_path, meta_path):
        transform = transforms.Compose(
            [transforms.ToTensor(),
             transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        self.image = Image.open(str(image_path)+".jpg").convert('RGB').resize((100,100))
        self.tensor = transform(self.image)
        self.meta = {node.tag: node.text for node in ET.parse(str(meta_path)).getroot().iter() }
        [self.meta.pop(key, None) for key in dict(self.meta) if '\n' in self.meta[key]]
        global classes
        self.index = self.set_label(classes)
    def set_label(self, classes):
#         print(self.meta['name'])
        for index, label in enumerate(classes):
            if self.meta['name'] == label:
                return index
        
def batch_generator(file_list, batch_size):
    n_total = len(file_list)
    n_images_left= len(file_list)
    n_images_used = 0
    assert n_images_left%batch_size == 0
    def _batch_generator():
        nonlocal n_total
        nonlocal n_images_left
        nonlocal n_images_used
        nonlocal batch_size
        if n_images_left == 0:
            return []
        else:
            n_images_left -= batch_size
            return_list = [labeled_image(file_list[n][0], file_list[n][1]) for n in range(n_images_used, n_images_used+batch_size)]
            n_images_used += batch_size
            return return_list
    return _batch_generator

def list_to_4tensor(tensor_list):
    b = torch.zeros(len(tensor_list), tensor_list[0].shape[0], tensor_list[0].shape[1], tensor_list[0].shape[2])
    for i in range(b.shape[0]):
        b[i] = tensor_list[i]
    return b

In [34]:
file_list = []
for directory in os.listdir(annotations_lib):
    for file in os.listdir(annotations_lib / directory):
        file_list += [(str(images_lib / directory / file), str(annotations_lib / directory / file))]
get_batch = batch_generator(file_list, 20)

In [35]:
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
trainloader = get_batch()
i=0
running_loss = 0
while len(trainloader) > 0:  # loop over the dataset multiple times

    running_loss = 0.0

    # get the inputs; data is a list of [inputs, labels]
    inputs = list_to_4tensor([data.tensor for data in trainloader])
    labels = torch.LongTensor([data.index for data in trainloader])
    # zero the parameter gradients
    optimizer.zero_grad()

    # forward + backward + optimize
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    # print statistics
    running_loss += loss.item()
    print(i, running_loss)
    trainloader = get_batch()
    i += 1
print('Finished Training')

C:\Users\rwzet\Miniconda3\envs\torch\lib\site-packages\ipykernel_launcher.py:23: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.


0 4.778354167938232
1 4.7769246101379395
2 4.7718186378479
3 4.764211654663086
4 4.759812355041504
5 4.748110771179199
6 4.740616321563721
7 4.742663860321045
8 4.7733893394470215
9 4.770295143127441
10 4.764841556549072
11 4.759216785430908
12 4.7484235763549805
13 4.7384033203125
14 4.727949619293213
15 4.720053672790527
16 4.721795082092285
17 4.812333583831787
18 4.808331489562988
19 4.800882339477539
20 4.796217441558838
21 4.784495830535889
22 4.7708234786987305
23 4.754448890686035
24 4.740438461303711
25 4.717505931854248
26 4.697209358215332
27 4.677476406097412
28 4.6603240966796875
29 4.695618629455566
30 4.774748802185059
31 4.771872043609619
32 4.761348724365234
33 4.7472405433654785
34 4.731270790100098
35 4.708643913269043
36 4.702550411224365
37 4.821093559265137
38 4.816771507263184
39 4.804211616516113
40 4.7888288497924805
41 4.769237041473389
42 4.744144439697266
43 4.713484287261963
44 4.684719085693359
45 4.644797325134277
46 4.606695175170898
47 4.665415763854980